

## Librarie's Installation



In [ ]:
!pip -q install -U "transformers==5.1.0" "accelerate>=1.0.0" datasets evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.7 MB/s eta 0:00:00


## Connection in google drive

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/sentiment_analysis_finetuning"
except Exception:
    SAVE_DIR = "."

os.makedirs(SAVE_DIR, exist_ok=True)
print("Results:", SAVE_DIR)

Mounted at /content/drive
Results: /content/drive/MyDrive/sentiment_analysis_finetuning


## Load IMDB dataset

In [ ]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")
print(imdb)
print("\nTrain size:", len(imdb["train"]), " ,  Test size:", len(imdb["test"]))
print("Example:", imdb["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Train size: 25000  ,  Test size: 25000
Example: {'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the

## Dataset split

In [ ]:
# Χωρίζουμε το train σε train (90%) + validation (10%)
split = imdb["train"].train_test_split(test_size=0.1, seed=42)
imdb["train"] = split["train"]
imdb["validation"] = split["test"]

print("  Train size:", len(imdb["train"]))
print("  Validation size:", len(imdb["validation"]))
print("  Test size:", len(imdb["test"]))

  Train size: 22500
  Validation size: 2500
  Test size: 25000


## Tokenization

In [ ]:
from transformers import AutoTokenizer

model_name_distil = "distilbert/distilbert-base-uncased"
tokenizer_distil = AutoTokenizer.from_pretrained(model_name_distil)

def tokenize_function(examples, tokenizer, max_length=512):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )

def tokenize_distil(examples):
    return tokenize_function(examples, tokenizer_distil)

imdb_distil = imdb.map(
    tokenize_distil,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing (DistilBERT)",
)
imdb_distil.set_format("torch")
print("Train:", len(imdb_distil["train"]), "| Validation:", len(imdb_distil["validation"]), "| Test:", len(imdb_distil["test"]))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing (DistilBERT):   0%|          | 0/22500 [00:00<?, ? examples/s]

Tokenizing (DistilBERT):   0%|          | 0/25000 [00:00<?, ? examples/s]

Tokenizing (DistilBERT):   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing (DistilBERT):   0%|          | 0/2500 [00:00<?, ? examples/s]

Train: 22500 | Validation: 2500 | Test: 25000


## Finetuning Training

In [ ]:
import os
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
from transformers import DataCollatorWithPadding
data_collator_distil = DataCollatorWithPadding(tokenizer=tokenizer_distil)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)


model_distil = AutoModelForSequenceClassification.from_pretrained(
    model_name_distil,
    num_labels=2,
)

training_args_distil = TrainingArguments(
    output_dir=os.path.join(SAVE_DIR, "results_distilbert"),
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    report_to="none",
)

trainer_distil = Trainer(
    model=model_distil,
    args=training_args_distil,
    train_dataset=imdb_distil["train"],
    eval_dataset=imdb_distil["validation"],
    data_collator=data_collator_distil,
    compute_metrics=compute_metrics,
)

trainer_distil.train()


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.283997,0.290195,0.910000
2,0.141184,0.318888,0.927200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=5626, training_loss=0.23727224867326105, metrics={'train_runtime': 665.7191, 'train_samples_per_second': 67.596, 'train_steps_per_second': 8.451, 'total_flos': 5961032939520000.0, 'train_loss': 0.23727224867326105, 'epoch': 2.0})

##Evaluation in test accuracy

In [ ]:
# Test accuracy
eval_distil = trainer_distil.evaluate(imdb_distil["test"])
print("DistilBERT - Test accuracy:", eval_distil["eval_accuracy"])

# Save metrics
import json
path = os.path.join(SAVE_DIR, "results_distil_metrics.json")
with open(path, "w") as f:
    json.dump({k: float(v) if hasattr(v, "item") else v for k, v in eval_distil.items()}, f, indent=2)
print("Saved", path)


DistilBERT - Test accuracy: 0.93104
Saved /content/drive/MyDrive/sentiment_analysis_finetuning/results_distil_metrics.json


## Evaluation on F1 metric

In [ ]:
import numpy as np
import evaluate

f1_metric = evaluate.load("f1")

def compute_f1_only(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]}

trainer_distil.compute_metrics = compute_f1_only
eval_f1_distil = trainer_distil.evaluate(imdb_distil["test"])
print("DistilBERT - Test F1:", eval_f1_distil["eval_f1"])


DistilBERT - Test F1: 0.9308463698355395
